In [1]:
from promptsmith.dspy_init import get_dspy
from promptsmith.tasks.bad_to_good.bulletize_text_2 import BulletizeText
from promptsmith.judges.judge_bullet_structure import JudgeBulletStructure
from promptsmith.judges.judge_coverage import JudgeCoverage
from promptsmith.judges.judge_focus_relevance import JudgeFocusRelevance
from promptsmith.judges.judge_redundancy import JudgeRedundancy
from promptsmith.refiners import Refiner
from promptsmith.evaluation.task_evaluator import TaskEvaluator
from promptsmith.utils.display import display_evaluation_result

dspy, lm = get_dspy()

/Users/yanivgal/dev/ai21/promptsmith/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
with open('../../data/text_01.txt', 'r') as f:
    text = f.read()

#### original text

In [3]:
print(text)

The first question in this study stated: What is the strength and direction of the relationship between faculty's perceptions of the importance of course design, communication, time management, and technical competency, and their ability to teach online? To address this question, descriptive statistics (mean and standard deviation) by individual competency items of each of the four competency dimensions: course design, course communication, time management, and technical competency,  were reported in Table 2 on page 23. Most of the items in each competency were rated high for both constructs, Importance, and Ability.
The items manage grades online (M = 4.73) and creating online assignments (M = 4.68)

were rated the highest in the course design competency. In the course communication competency, responding to student questions promptly (M = 4.79) and providing feedback on assignments (M = 4.65) were rated the highest. In time management, scheduling time to design the course prior to de

#### bulletizing

In [4]:
bulletize = dspy.ChainOfThought(BulletizeText)
bulletized = bulletize(input_text=text)

In [5]:
print(bulletized.output_text)

# Faculty Perceptions of Online Teaching Competencies

The study investigates faculty perceptions of online teaching competencies and their relationships with demographic factors.

## Research Question 1: Relationship Between Perceptions and Ability
**What is the strength and direction of the relationship between faculty's perceptions of the importance of competencies and their ability to teach online?**
- Descriptive statistics (mean and standard deviation) reported in Table 2 on page 23.
- High ratings for both constructs, Importance and Ability, across competencies.
  - **Course Design:**
    - Manage grades online (M = 4.73)
    - Creating online assignments (M = 4.68)
  - **Course Communication:**
    - Responding to student questions promptly (M = 4.79)
    - Providing feedback on assignments (M = 4.65)
  - **Time Management:**
    - Scheduling time to design the course prior to delivery (M = 4.65)
    - Spending weekly hours to grade assignments (M = 4.54)
  - **Technical Compet

#### judging

In [ ]:
evaluator = TaskEvaluator(
    task=dspy.ChainOfThought(BulletizeText),
    judges={
        'structure': dspy.Predict(JudgeBulletStructure),
        'coverage': dspy.Predict(JudgeCoverage),
        'focus_relevance': dspy.Predict(JudgeFocusRelevance),
        'redundancy': dspy.Predict(JudgeRedundancy)
    },
    weights={
        'structure': 0.5,
        'coverage': 0.2,
        'focus_relevance': 0.15,
        'redundancy': 0.15,
    }
)

# Evaluate
result = evaluator.evaluate(text)
print(f"Combined score: {result.combined_score:.2f}")

# Check against threshold
if result.passed(threshold=0.88):
    print("All checks passed!")
else:
    print("Some checks failed.")

Combined score: 0.85
Some checks failed.


In [9]:
display_evaluation_result(result)


[EVALUATION]
Combined score: 0.85

structure: 0.70
  - The title is appropriate and meets the length requirement.
- The one-line summary is present and concise.
- Each logical block begins with a meaningful H2 heading.
- The bullet points are well-structured, but there are more than 8 top-level bullets in the first section, which violates the guideline.
- Each section has a closing line, but the closing line in the first section does not follow the required format of being italicized.
- There are three dashes separating the major sections.
- There is no extraneous text outside the outline.

Overall, the output has a few structural issues, particularly with the number of top-level bullets and the format of the closing line.

coverage: 1.00
  The output text effectively captures the key ideas from the original input. It maintains the structure of the research questions and summarizes the findings related to faculty perceptions of online teaching competencies. All major points, including